# Lesson 1: 🔍 Search for floats 

### 🎯 Learning Objectives
In this lesson, we will use the Argo index file to search for the floats and profiles we want based on our research interests. Specifically, we will do this by filtering the index file by considering several factors:

* BGC parameters (e.g., we want profiles that include both nitrate and PAR)
* geographical range (e.g., we want profiles within the northwest Pacific)
* temporal range (e.g., we want profiles collected during 2024)
* sampling duration (e.g., we want floats that have collected profiles at least for two years)
* sampling frequency (e.g., we want floats that have collected profiles at least once per month)
* drift speed (e.g., we want floats that have drifted very slowly)

### 🛠️ Prerequisites
Before starting this lesson, you should have a brief understanding of what the BGC-Argo program is all about. For this, we recommend reading [Bittig et al. (2019)](https://www.frontiersin.org/article/10.3389/fmars.2019.00502/full). In addition, we recommend reading about the structure and format of the BGC-Argo data, which are nicely summarized on this [Data FAQ](https://www.go-bgc.org/data/data-faq) page provided by GO-BGC (Global Ocean Biogeochemistry Array).

### ❓ How to Use This Notebook
* 📚 **Read** the tutorial text blocks carefully, as they provide the essential background information behind the code.
* ▶️ **Run** each code cell sequentially by clicking the cell and pressing `Shift + Enter`.
* 📝 **Exercise** your knowledge! At the end of this notebook, we provide active learning exercises where you will need to write the code yourself.

### 🚀 Ready? Let's Get Started!
---

## 📚 Tutorials

### Import libraries

▶️ Run the cell below to import relevant Python libraries.

In [ ]:
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cft 
from datetime import datetime, timedelta

☝️ If no error messages are shown, the import was successful. 

### The index file

The original BGC-Argo data (profiles and time series) are hosted by the Global Data Assembly Center (GDAC) which has two nodes, namely [CORIORIS](https://data-argo.ifremer.fr) and [US-GODAE](https://usgodae.org/pub/outgoing/argo). To search and identify BGC-Argo floats and profiles, we will use the Argo index file for synthetic profiles (called [Sprof files](https://www.go-bgc.org/data/data-faq)). Click either [the CORIORIS copy](https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt) or [the US-GODAE copy](https://usgodae.org/pub/outgoing/argo/argo_synthetic-profile_index.txt) to see the actual contents of the latest index file.

The index file contains the list of all available synthetic profiles collected by all kinds of Argo floats (i.e., Core, BGC, and Deep). It gives us the basic information for each profile, including when (**date**) and where (**latitude** and **longitude**) the profile was collected and which variables (**parameters**) were measured.

▶️ Run the cell below to load the index file. 
* `skiprows` is used to ignore the first 8 rows of the index file (which are just comments).
* `usecols` is used to load only the selected columns (i.e., file, date, latitude, longitude, and parameters).
* We will first try loading the index file from the CORIORIS node. If the node is down for some reason (e.g., maintenance), we will use the US-GODAE node.
* **Note** that the loading can take up to a minute or so depending on your interenet speed because the index file is relatively large (e.g., 57 MB as of March 17, 2026).

In [ ]:
try:
    # 1. Try the French server with a strict 10-second timeout
    print("Connecting to the French GDAC...")
    url1 = urllib.request.urlopen('https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt', timeout=10)
    
    df_index = pd.read_csv(url1, 
                           skiprows=8, 
                           usecols=['file', 'date', 'latitude', 'longitude', 'parameters'])
    print("✅ Success!")

except Exception:
    # 2. If it stalls or fails, immediately fall back to the US server
    print("⚠️ French server timed out. Trying the US GDAC...")
    url2 = urllib.request.urlopen('https://usgodae.org/pub/outgoing/argo/argo_synthetic-profile_index.txt', timeout=10)
    
    df_index = pd.read_csv(url2, 
                           skiprows=8, 
                           usecols=['file', 'date', 'latitude', 'longitude', 'parameters'])
    print("✅ Success!")

df_index

☝️ We see a spreadsheet showing the first and last five rows. Each row represents a profile collected by a float. The total number of rows represents the total number of profiles collected by the entire Argo program. How many profiles are there in the latest index file? As of March 19, 2026, there were 384,050 profiles.

We will use this index file as a starting point and reduce the number of profiles as we filter by different criteria.

### Filter by BGC parameters

We will start filtering by BGC parameters. First, let's find out which parameters are available in the column **parameters**. 

▶️ Run the cell below to show the list of available parameters in the index file.

In [ ]:
unique_params = df_index['parameters'].str.split().explode().unique()
print(*unique_params, sep='\n')

☝️ **PRES**, **TEMP**, and **PSAL** are the pressure (proxy for depth), temperature, and salinity, respectively. In principle, all profiles should contain these three physical parameters.

The rest are mostly biogeochemical parameters. As described in [Bittig et al. (2019)](https://www.frontiersin.org/article/10.3389/fmars.2019.00502/full), the following are the standard six BGC parameters:

* **CHLA**: chlorophyll-a (mg/m3)
* **BBP700**: back scattering coefficient at 700 nm (m-1)
* **NITRATE**: nitrate (umol/kg)
* **DOWNWELLING_PAR**: photosynthetically active radiation (W/m2)
* **DOXY**: dissolved oxygen (umol/kg)
* **PH_IN_SITU_TOTAL**: pH (-)

When a float is equipped with sensors that can measure all these six parameters, it is referred to as a **full-sensor** float. In this tutorial series, we will work with these six parameters only.

▶️ Run the cell below to define the BGC parameters we want. Here let's choose (1) dissolved oxygen and (2) back scattering coefficient at 700 nm.

In [ ]:
bgc_parameters = ["DOXY", "BBP700"]

▶️ Run the cell below to filter the index file by the selected BGC parameters.

In [ ]:
# Make a copy of the original index file as a starting point
df_index_param = df_index.copy()

# Loop through each parameter
for param in bgc_parameters:
    # Filter the dataframe and overwrite it with the smaller, filtered version
    df_index_param = df_index_param[df_index_param['parameters'].str.contains(param)]

# Display the output
df_index_param

☝️ Now the index file should only contain profiles that have both **DOXY** and **BBP700**. Did you notice that the number of rows (profiles) has decreased after this filtering?

### Filter by longitudes, latitudes, and dates

Next, we will filter the index file based on our spatial and temporal interests.

▶️ Run the cell below to define the spatial domain and temporal coverage we want:

* **lon0**: the easternmost longitude of the spatial domain (float: -180 to 180; **HOWEVER**, use 0 to 360 if the zonal range crosses the International Date Line).
* **lon1**: the westernmost longitude of the spatial domain (float: -180 to 180; **HOWEVER**, use 0 to 360 if the zonal range crosses the International Date Line. Make sure that **lon1** > **lon0**).
* **lat0**: the southernnmost latitude of the spatial domain (float: -90 to 90).
* **lat1**: the northernmost latitude of the spatial domain (float: -90 to 90. Make sure that **lat1** > **lat0**).
* **date0**: the first date of the temporal coverage (string: 'YYYY-MM-DD').
* **date1**: the last date of the temporal coverage (string: 'YYYY-MM-DD'. Make sure that **date1** > **date0**).

Here, we will select the northwest Pacific (117-150E and 17-50N) during 2023-2024.

In [ ]:
lon0 = 117
lon1 = 150
lat0 = 17
lat1 = 50
date0 = '2023-01-01'
date1 = '2024-12-31'

▶️ Run the cell below to filter the index file by the selected longitudes and latitudes.

In [ ]:
# Check
if lon0 > lon1:
    raise ValueError(f"lon0 cannot be greater than lon1.")

# Filter by longitudes
if lon0 > 180 or lon1 > 180:
    if lon0 < 0 or lon1 < 0:
        raise ValueError(
            f"lon0 and lon1 cannot be a mixture of <0 and >180." 
            f" You entered: {lon0} and {lon1}."
            f" Use either the -180 to 180 system or the 0 to 360 system."
        )
    else:
        # for the 0 to 360 system
        df_index_lon = df_index_param[(df_index_param['longitude'] % 360 >= lon0) & (df_index_param['longitude'] % 360 <= lon1)] 
else:
    # for the -180 to 180 system
    df_index_lon = df_index_param[(df_index_param['longitude'] >= lon0) & (df_index_param['longitude'] <= lon1)] 

if lat0 > lat1:
    raise ValueError(f"lat0 cannot be greater than lat1.")

# Filter by latitudes
df_index_lat = df_index_lon[(df_index_lon['latitude'] >= lat0) & (df_index_lon['latitude'] <= lat1)]

# Display the result
df_index_lat

☝️ Notice that the number of rows (profiles) has decreased after the filtering.

▶️ Run the cell below to filter the index file by the selected dates.

In [ ]:
# Check
if date0 > date1:
    raise ValueError(f"date0 cannot be greater than date1.")

# For some reason, there are profiles with missing dates. Remove them first.
df_index_date = df_index_lat[df_index_lat['date'] >= 0]

# Convert date from float64 to string
df_index_date['date_str'] = df_index_date['date'].astype('int64').astype(str)

# Convert the string to a proper datetime object, which is easy to work with in Python.
df_index_date['datetime'] = pd.to_datetime(df_index_date['date_str'], format='%Y%m%d%H%M%S')

# Filter the index based on the temporal interest
df_index_date = df_index_date[(df_index_date['datetime'] >= f'{date0} 00:00:00') & (df_index_date['datetime'] <= f'{date1} 23:59:59')]

# Delete the columns `date` and `date_str` which are no longer needed
del df_index_date['date']
del df_index_date['date_str']

# Print
df_index_date

☝️ Notice that the number of rows (profiles) has decreased after the filtering.

### WMO: The unique identifier for Argo floats

All Argo floats are given unique seven-digit integers called the WMO (World Meteorological Organization) IDs, so that we can identify them easily. In the original Argo index file, this ID (which we will refer to as **wmo** for simplicity) is provided as the name for the second parental directory for the column **file**. This structure is a bit difficult to access wmo, so we will extract this information and create a new column called **wmo**.

▶️ Run the cell below to add **wmo** as an additional column.

In [ ]:
# Make a copy of the filtered index file
df_index_wmo = df_index_date.copy()

# Split the file string by slashes and take the second item corresponding to wmo
df_index_wmo['wmo'] = df_index_wmo['file'].str.split('/').str[1]

# Display the output
df_index_wmo

☝️ There we have the new column **wmo**!

### Functions

So far, we learned how to (1) load the index file, (2) filter it by BGC parameters, longitudes, latitudes, and dates; and (3) add wmo as an additional parameter. If we want to repeat these steps with different inputs in the future, it would be convenient to create **functions**.

▶️ Run the cell below to define (`def`) a function that loads the original Argo index file from one of the GDAC nodes. We name this function `load_index`.

In [ ]:
def load_index():
    """
    This function attempts to load the Argo synthetic-profile index file from one of the two GDAC nodes.

    INPUT:
    * None

    OUTPUT:
    * df_index: the latest index file in the pandas dataframe format 
    """
    
    try:
        # 1. Try the French server with a strict 10-second timeout
        print("Connecting to the French GDAC...")
        url1 = urllib.request.urlopen('https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt', timeout=10)
        
        df_index = pd.read_csv(url1, 
                               skiprows=8, 
                               usecols=['file', 'date', 'latitude', 'longitude', 'parameters'])
        print("✅ Success!")
    
    except Exception:
        # 2. If it stalls or fails, immediately fall back to the US server
        print("⚠️ French server timed out. Trying the US GDAC...")
        url2 = urllib.request.urlopen('https://usgodae.org/pub/outgoing/argo/argo_synthetic-profile_index.txt', timeout=10)
        
        df_index = pd.read_csv(url2, 
                               skiprows=8, 
                               usecols=['file', 'date', 'latitude', 'longitude', 'parameters'])
        print("✅ Success!")
    
    return df_index

▶️ Run the cell below to load the index file using this function.

In [ ]:
df_index = load_index()
df_index

☝️ Did you notice that we got the same result as we loaded the index at the beginning of this tutorial?

### The function `filter_index`

▶️ Run the cell below to define a function that filters the index file by BGC parameters, longitudes, latitudes, and dates all at once.

In [ ]:
def filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters):
    """
    This function filters the index file by BGC parameters, longitudes, latitudes, and dates.
    
    INPUT:
    * df_index: the original index file, which can be obtained by `load_index()`
    * lon0, lon1, lat0, lat1: the east, west, south, and north edges of the spatial domain
    * date0, date1: the beginning and end of the temporal coverage (string: 'YYYY-MM-DD')
    * bgc_parameters: the list of BGC parameters (string or list)

    OUTPUT:
    * df_index_out: the filtered index file
    """
    
    # Filter by parameters
    df_index_out = df_index.copy()
    for param in bgc_parameters:
        df_index_out = df_index_out[df_index_out['parameters'].str.contains(param)]
    
    # Filter by longitudes    
    if lon0 > lon1:
        raise ValueError(f"lon0 cannot be greater than lon1.")
    else:
        if lon0 > 180 or lon1 > 180:
            if lon0 < 0 or lon1 < 0:
                raise ValueError(
                    f"lon0 and lon1 cannot be a mixture of <0 and >180." 
                    f" You entered: {lon0} and {lon1}."
                    f" Use either the -180 to 180 system or the 0 to 360 system."
                )
            else:
                # for the 0 to 360 system
                df_index_out = df_index_out[(df_index_out['longitude'] % 360 >= lon0) & (df_index_out['longitude'] % 360 <= lon1)] 
        else:
            # for the -180 to 180 system
            df_index_out = df_index_out[(df_index_out['longitude'] >= lon0) & (df_index_out['longitude'] <= lon1)] 
    
    # Filter by latitudes
    if lat0 > lat1:
        raise ValueError(f"lat0 cannot be greater than lat1.")
    else:
        # Filter by latitudes
        df_index_out = df_index_out[(df_index_out['latitude'] >= lat0) & (df_index_out['latitude'] <= lat1)]

    # Check dates
    if date0 > date1:
        raise ValueError(f"date0 cannot be greater than date1.")
    
    # For some reason, there are profiles with missing dates. Remove them first.
    df_index_out = df_index_out[df_index_out['date'] >= 0]
    
    # Convert date from float64 to string
    df_index_out['date_str'] = df_index_out['date'].astype('int64').astype(str)
    
    # Convert the string to a proper datetime object
    df_index_out['datetime'] = pd.to_datetime(df_index_out['date_str'], format='%Y%m%d%H%M%S')
    
    # Filter by dates
    df_index_out = df_index_out[(df_index_out['datetime'] >= f'{date0} 00:00:00') & (df_index_out['datetime'] <= f'{date1} 23:59:59')]
    
    # Delete the columns `date` and `date_str` which are no longer needed
    del df_index_out['date']
    del df_index_out['date_str']

    # Add WMO
    df_index_out['wmo'] = df_index_out['file'].str.split('/').str[1]
    
    return df_index_out

☝️ As defined, this function requires the following inputs (arguments):

* **df_index**: the original Argo index file, which can be loaded with `load_index()`.
* **lon0, lon1, lat0, lat1, date0, date1, bgc_parameters**, all of which we defined earlier.

▶️ Run the cell below to filter the index based on our previously-defined inputs and return the filtered index. 

In [ ]:
df_index_filtered = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
df_index_filtered

☝️ Notice that the result is identical as earlier.

If you forget what the function does, you can use `help()` to see the description.

▶️ Run the cell below to see the description for `filter_index`.

In [ ]:
help(filter_index)

### Filter by sampling duration, frequency, and drift speed

Sometimes, we are looking for floats that have collected profiles for specific duration or at specific frequency. Furthermore, we may be interested in floats that are relatively stationary, which are useful for time-series studies and vertical one-dimensional ocean modelling. 

We will consider these factors to further filter the index. Specifically, we define the following three additional inputs:

* **mindura**: the minimum duration of the sampling (in days)
* **minfreq**: the minimum frequency of the profiling cycle (in days)
* **maxdrif**: the maximum drift speed (in m/s). For learning more about Argo floats' drift, see [Katsumata and Yoshinari (2010)](https://doi.org/10.1007/s10872-010-0046-4).

▶️ Run the cell below to define these three criteria.

In [ ]:
mindura = 100
minfreq = 14
maxdrif = 0.05

☝️ In this example, we want floats that have collected profiles for at least 100 days once every two weeks and drifting at a rate no more than 0.05 m/s (which is considered relatively slow).

### The function `calculate_float_speed`

▶️ Run the cell below to define a function that calculates the mean drift of a float based on longitudes, latitudes, and dates (technially, dates and times).

In [ ]:
def calculate_float_speed(lon, lat, date):
    """
    This function calculates a rough estimate for the float's drift speed at parking depth, which is typically at 1,000 m.
    NOTE that this represents the overall mean speed and not the mean of the instantaneous speed (between cycles). 
    This prevents from division by zero when there are multiple sampling (especially in the first day of deployment).

    INPUT:
    * lon: longitudes of the profiles (array)
    * lat: latitiudes of the profiles (array)
    * date: dates of the profiles (array)

    OUTPUT:
    * speed: the roughly estimated drift speed of the float (m/s)
    """
    
    # Earth's radius in meters
    R = 6371000  
    
    # convert to radians
    lon_rad = np.radians(lon)
    lat_rad = np.radians(lat)
    
    # take the difference
    delta_lon = lon_rad.diff()
    delta_lat = lat_rad.diff()
    
    # Haversine formula
    a = np.sin(delta_lat / 2)**2 + np.cos(lat_rad[:-1]) * np.cos(lat_rad[1:]) * np.sin(delta_lon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distances = R * c  # in meters
    
    # Speed in meters per second
    speed = distances.sum() / date.diff().sum().total_seconds()
    
    # average speed
    return speed

### The function `filter2_and_save_index`

▶️ Run the cell below to define a function that filters the index by the three criteria and save the filtered index file as [a Parquet file](https://www.datacamp.com/tutorial/what-is-a-parquet-file). Note that the three criteria are optional (if we do not care about these criteria, we do not need to provide them as input arguments). Also note that `note` is also an optional argument, which is used to create a unique file name for the filtered index file.

In [ ]:
def filter2_and_save_index(df_index, mindura=None, minfreq=None, maxdrif=None, note='default'):
    """
    This function filters the index file by the duration and frequency of the sampling as well as the drift speed and saves the filtered index as a CSV file.

    INPUT:
    * df_index: the index file (filtered by BGC parameters, longitudes, latitudes, and dates using `filter_index`) (format: pandas dataframe)
    * mindura: (format: integer)
    * minfreq: (format: integer)
    * maxdrif: (format: float)
    * note: this will be added to the file name (format: string)

    OUTPUT:
    * df_filtered2: (format: pandas dataframe). This will also be saved as a CSV file.
    """
    
    # Create a copy to create a further filtered index
    df_filtered2 = df_index.copy()
    
    for wmo, group in df_filtered2.groupby('wmo'):
        
        # Sort the group by time just to be safe
        group = group.sort_values('datetime')
    
        # Keep the profile if there is only one profile and mindura is set to 1 or None
        if len(group) == 1 and mindura in (None, 0, 1):
            valid_dura = True
            valid_freq = True
            valid_drif = True
        
        else: 
            # Filter out the float with a single profile
            if len(group) == 1:
                df_filtered2 = df_filtered2[df_filtered2['wmo'] != wmo]
            # Check the three criteria for multiple profiles
            else:  
                if mindura is None:
                    valid_dura = True
                else:
                    valid_dura = (group['datetime'].max() - group['datetime'].min()) > pd.to_timedelta(mindura, unit='D')
                if minfreq is None:
                    valid_freq = True
                else:
                    # Check the minimum frequency
                    valid_freq = group['datetime'].diff().max() < pd.to_timedelta(minfreq, unit='D')
                if maxdrif is None:
                    valid_drif = True
                else:
                    valid_drif = calculate_float_speed(group['longitude'], group['latitude'], group['datetime']) < maxdrif
    
        # Filter out the float's profiles if not true that all three are true
        if not (valid_dura and valid_freq and valid_drif):
            df_filtered2 = df_filtered2[df_filtered2['wmo'] != wmo]
    
    filename = f"../index_files/argo_synthetic-profile_index_{note}.parquet"
    df_filtered2.to_parquet(filename)

    print(f'\n{filename} has been saved...')
    return df_filtered2

▶️ Run the cell below to use this function.

In [ ]:
filter2_and_save_index(df_index_filtered, mindura, minfreq, maxdrif)

☝️ Notice that the number of profiles has decreased after this filtering. Also check your directory to make sure that this filtered index has been saved as a CSV file in the directory **index_files** which is located in the root directory (in the same directory as **lessons**). This is the final filtered index file and we will use it as a basis for our next lessons.

**This is the end of the tutorials for this lesson. Hope you enjoyed it!**

---

## 📝 Exercises

Now that we have learned how to use the index file to search for floats and profiles based on our research interests, let's do a few exercises by considering different use cases. Use the three functions developed for convenience (`load_index`, `filter_index`, and `filter2_and_save_index`).

### Exercise 1: Create and save a filtered index file that provides a list of profiles that measured chlorophyll-a on March 1, 2026 in the global ocean.

**NOTE**: When saving the index file using `filter2_and_save_index`, use `note = 'ex1'` provided below as the argument for `note`, so that it will not overwrite the CSV file created in the tutorial.

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

lon0 = -180
lon1 = 180
lat0 = -90
lat1 = 90
date0 = '2026-03-01'
date1 = '2026-03-01'
bgc_parameters = 'CHLA'
mindura = None
minfreq = None
maxdrif = None
note = 'ex1'
df_index = load_index()
df_filter = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
filter2_and_save_index(df_filter, mindura, minfreq, maxdrif, note)

### Exercise 2: Create and save a filtered index file that only contains profiles collected by the *full-sensor* BGC-Argo floats in the Indian Ocean at least once per month and at least for a year between 2018 and 2025.

Set `note = 'ex2'` for `filter2_and_save_index`.

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

The longitude and latitude coordinates for the Indian Ocean are arbitrary.
```python

lon0 = 25
lon1 = 115
lat0 = -40
lat1 = 10
date0 = '2018-01-01'
date1 = '2025-12-31'
bgc_parameters = ['DOXY', 'NITRATE', 'CHLA', 'BBP700', 'PH_IN_SITU_TOTAL', 'DOWNWELLING_PAR']
mindura = 365
minfreq = 30
maxdrif = None
note = 'ex2'
df_index = load_index()
df_filter = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
filter2_and_save_index(df_filter, mindura, minfreq, maxdrif, note)

### Exercise 3: Create and save a filtered index file that only contains profiles that measured both dissolved oxygen and nitrate and drifted at a speed of nore more than 0.05 m/s in the North Pacific between 2019 and 2022.
**TIP**: Be careful how you define the longitude range, as it crosses the over the International Date Line.

Set `note = 'ex3'` for `filter2_and_save_index`.

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

The longitude and latitude coordinates for the North Pacific are arbitrary.
```python

lon0 = 120
lon1 = 260
lat0 = 0
lat1 = 60
date0 = '2019-01-01'
date1 = '2022-12-31'
bgc_parameters = ['DOXY', 'NITRATE']
mindura = None
minfreq = None
maxdrif = 0.05
note = 'ex3'
df_index = load_index()
df_filter = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
filter2_and_save_index(df_filter, mindura, minfreq, maxdrif, note)


---

This is the end of the lesson. If you are using **Binder**, don't forget to dowload the files created in this lesson before you lose connection!

Well done 🎉, take a break 💤, have another cup ☕, and move to the next lesson ✍️ when you are ready 💪

While your memory is fresh, please feel free to provide your user experience on this lesson by visiting [this link](https://forms.gle/oAGmz5RTW4Pp46bt7). Thanks!